# ARC Cluster Inspector

Displays all tasks in a cluster: training pair grids (input → output) + stored LLM description.

**Usage:** Run cells 1–3 once, then change `CLUSTER` in cell 3 and re-run cells 3 and 4.

In [ ]:
%matplotlib inline
import json, re
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

# ARC 10-colour palette
ARC_COLORS = ['#000000','#0074D9','#FF4136','#2ECC40','#FFDC00',
               '#AAAAAA','#F012BE','#FF851B','#7FDBFF','#870C25']
CMAP = mcolors.ListedColormap(ARC_COLORS)
NORM = mcolors.BoundaryNorm(range(11), CMAP.N)

def _find_project_root() -> Path:
    p = Path.cwd()
    for _ in range(8):
        if (p / 'data' / 'training').exists():
            return p
        if (p / 'CLAUDE.md').exists():
            return p
        p = p.parent
    raise FileNotFoundError(
        'Cannot find project root. '
        'Run: jupyter notebook from inside the arc-agi-solver directory.'
    )

PROJECT = _find_project_root()
print(f"Project root: {PROJECT}")

In [ ]:
def parse_inspection(path: Path) -> dict:
    """Parse cluster_inspection.txt into {cluster_id: {task_id: description}}."""
    text = path.read_text()
    result = {}

    # Each cluster block: header line, then task entries until next header or EOF
    cluster_pat = re.compile(
        r'={60}\nCluster (\d+) \(n=\d+\)\n={60}\n(.*?)(?=\n={60}\nCluster |\Z)',
        re.DOTALL,
    )
    task_pat = re.compile(
        r'  ([0-9a-f]{8}):\n(.*?)(?=\n  [0-9a-f]{8}:|\Z)',
        re.DOTALL,
    )

    for m in cluster_pat.finditer(text):
        cid   = int(m.group(1))
        body  = m.group(2)
        tasks = {}
        for tm in task_pat.finditer(body):
            tasks[tm.group(1)] = tm.group(2).strip()
        result[cid] = tasks

    return result

cluster_data = parse_inspection(PROJECT / 'results' / 'cluster_inspection.txt')
print(f"Parsed {len(cluster_data)} clusters: {sorted(cluster_data.keys())}")

In [ ]:
# ── Change CLUSTER to inspect a different cluster ────────────────────────
CLUSTER = 9

task_desc = cluster_data[CLUSTER]
task_ids  = sorted(task_desc.keys())

tasks = {}
for tid in task_ids:
    p = PROJECT / 'data' / 'training' / f'{tid}.json'
    if p.exists():
        tasks[tid] = json.loads(p.read_text())
    else:
        print(f'  Warning: {tid}.json not found')

print(f'Cluster {CLUSTER}: {len(tasks)} / {len(task_ids)} tasks loaded')

In [ ]:
def show_grid(ax, grid, title):
    ax.imshow(np.array(grid), cmap=CMAP, norm=NORM, interpolation='nearest')
    ax.set_title(title, fontsize=8)
    ax.axis('off')


for tid in task_ids:
    if tid not in tasks:
        print(f'── {tid}  (file not found) ──')
        continue

    task  = tasks[tid]
    desc  = task_desc[tid]
    pairs = task['train']
    test  = task['test'][0]

    n_rows = len(pairs) + 1   # train pairs + test row

    print(f"\n{'─'*62}")
    print(f'  {tid}   ({len(pairs)} training pairs)')
    print(f"{'─'*62}")

    fig, axes = plt.subplots(n_rows, 2, figsize=(6, 2.5 * n_rows), squeeze=False)

    for i, pair in enumerate(pairs):
        show_grid(axes[i, 0], pair['input'],  f'Train {i+1}  input')
        show_grid(axes[i, 1], pair['output'], f'Train {i+1}  output')

    show_grid(axes[-1, 0], test['input'], 'Test  input')
    if 'output' in test:
        show_grid(axes[-1, 1], test['output'], 'Test  output')
    else:
        axes[-1, 1].axis('off')
        axes[-1, 1].set_title('Test output (hidden)', fontsize=8)

    plt.tight_layout()
    plt.show()

    print(desc)
    print()